# Scale one training run across many devices

You will take the language model from the previous notebook and run it two ways on eight devices: with replicated parameters (data parallelism, `fsdp_size=1`) and with fully sharded parameters (`fsdp_size=8`). You will print where one weight actually lives in each layout, compare step time and device memory between the two, and move a checkpoint written under one mesh onto the other.

There is no second code path to learn. The trainer builds a two-axis mesh named `(data, fsdp)`, derives a sharding for every leaf of the train state, and hands both to `jax.jit`; XLA inserts the collectives. `fsdp_size=1` is the degenerate mesh with one device on the `fsdp` axis, which is plain data parallelism, and it needs no separate branch.

**Where this runs.** A Colab TPU runtime has 8 chips on one host and runs this notebook as written. Any machine can fake the same 8 devices with `--xla_force_host_platform_device_count=8`; the `SIMULATE_DEVICES` flag in the parameters cell sets that before JAX loads, so the notebook is runnable on a CPU laptop. On a single GPU both meshes degenerate to that one device and the notebook still runs, but the comparison becomes one device against itself.

**Expected time.** About 10 minutes on a TPU v5e-8 (two 60-step fits plus compilation) and about 30 to 45 minutes through the simulated-devices path, most of it XLA compilation.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
import os

SIMULATE_DEVICES = True  # fake 8 host devices (CPU) so the notebook runs anywhere; set False on real hardware
STEPS = 60               # per fit; enough for a stable step time, not a language model
BATCH_SIZE = 64
SEQUENCE_LENGTH = 256
EMB_FEATURES = 512
NUM_LAYERS = 12
NUM_HEADS = 8

if SIMULATE_DEVICES:
    # XLA reads this variable once, when it opens a backend, so it has to be set
    # before the first JAX import anywhere in the process.
    os.environ["XLA_FLAGS"] = (
        os.environ.get("XLA_FLAGS", "")
        + " --xla_force_host_platform_device_count=8").strip()

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The mesh

`build_mesh(fsdp_size)` returns a mesh of shape `(device_count // fsdp_size, fsdp_size)` with the axes named `data` and `fsdp`. Batches are split across every device on both axes at once; only the parameters distinguish the axes. On an 8-chip host, `fsdp_size=1` is an `(8, 1)` mesh where every device holds the whole model and a different slice of the batch, and `fsdp_size=8` is a `(1, 8)` mesh where every device sees an eighth of the batch and an eighth of every large parameter.

In [ ]:
from dew.training.distributed import build_mesh

mesh_dp = build_mesh(1)
mesh_fsdp = build_mesh(8)
print("data-parallel mesh:", mesh_dp.shape)
print("fsdp mesh:", mesh_fsdp.shape)

## Where a parameter lives

`parameter_spec(shape, fsdp_size, min_shard_size)` is the rule. The largest axis that divides evenly by `fsdp_size` is split over the `fsdp` axis, and anything below `fsdp_min_param_size` elements (65536 by default) stays replicated, because below that a parameter costs more in collectives than it saves in memory. Optimizer moments and the EMA copy have the same shapes as the parameters they track, so they pick up the same spec without anyone describing the optimizer's layout.

The visualisations below show the embedding table (256 rows of the byte vocabulary, 512 columns) under both meshes. Row one is the whole table on every device. Row two is the same table split eight ways over `fsdp`.

In [ ]:
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec

weight = jnp.ones((256, 512))
print("fsdp_size=1: every device holds the whole table")
jax.debug.visualize_array_sharding(
    jax.device_put(weight, NamedSharding(mesh_dp, PartitionSpec())))
print("fsdp_size=8: the row axis splits eight ways")
jax.debug.visualize_array_sharding(
    jax.device_put(weight, NamedSharding(mesh_fsdp, PartitionSpec("fsdp"))))

## The run

The data is Tiny Shakespeare again, tokenized in the notebook exactly as in the language model one. Then two trainers are built over the same model and objective, one per mesh, and each runs `STEPS` steps. The trainer prints the mesh it built, and the state is initialised directly into the target layout, so a model too large for one device is never materialised on one device.

In [ ]:
import json
import urllib.request
from pathlib import Path

import numpy as np
from dew.data.text import ByteTokenizer

DATA_DIR = Path("shakespeare-scaling")
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw = DATA_DIR / "shakespeare.txt"
if not raw.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        raw)
tokenizer = ByteTokenizer()
ids = np.asarray(tokenizer.encode(raw.read_text(encoding="utf-8")))
val_len = int(round(len(ids) * 0.02))
ids[:val_len].astype("uint8").tofile(DATA_DIR / "val.bin")
ids[val_len:].astype("uint8").tofile(DATA_DIR / "train.bin")
meta = {"tokenizer": "byte", "vocab_size": 256, "dtype": "uint8",
        "train_tokens": len(ids) - val_len, "val_tokens": val_len}
(DATA_DIR / "meta.json").write_text(json.dumps(meta))
print(meta)

from dew.data.dataloaders import get_token_dataset_grain

data = get_token_dataset_grain(
    str(DATA_DIR / "train.bin"), str(DATA_DIR / "val.bin"),
    batch_size=BATCH_SIZE, seq_len=SEQUENCE_LENGTH, worker_count=2)

In [ ]:
import optax
from dew.objectives.lm import LMObjective
from dew.registry import apply_precision_policy, build_model

model_config = apply_precision_policy("causal_transformer", dict(
    vocab_size=meta["vocab_size"], emb_features=EMB_FEATURES,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
    max_seq_len=SEQUENCE_LENGTH,
), dtype="bfloat16", attention_impl="auto")
model = build_model("causal_transformer", model_config)
objective = LMObjective(model, SEQUENCE_LENGTH, vocab_size=meta["vocab_size"])

In [ ]:
import time
from dew.training import ObjectiveTrainer


def make_trainer(fsdp_size, name, load_from=None):
    return ObjectiveTrainer(
        model, optax.adamw(1e-3), objective=objective, input_config=None,
        rngs=jax.random.PRNGKey(0), name=name, fsdp_size=fsdp_size,
        checkpoint_base_path="./checkpoints/scaling",
        load_from_checkpoint=load_from)


def run(trainer, steps):
    start = time.time()
    trainer.fit(data, training_steps_per_epoch=steps, epochs=1, val_steps_per_epoch=0)
    return time.time() - start


def device_memory():
    stats = jax.devices()[0].memory_stats()
    return stats.get("bytes_in_use", 0) if stats else 0


dp_trainer = make_trainer(1, "lm-replicated")
fsdp_trainer = make_trainer(8, "lm-sharded")

# 12L x 512W with tied embeddings: 42M parameters, three times over with
# the optimizer moments and the EMA copy.
params = jax.eval_shape(objective.init_params, jax.random.PRNGKey(0))
print(f"{sum(x.size for x in jax.tree_util.tree_leaves(params)) / 1e6:.1f}M parameters")

## The comparison

Both fits run the same steps on the same data; the only difference is the mesh. `bytes_in_use` is what one device holds after its fit, parameters and optimizer moments and EMA included. On real hardware the sharded run holds about an eighth of each large parameter and the replicated run holds all of them; through simulated CPU devices the split is real in the layout, and the step time is dominated by XLA's simulation of the collectives, so read the memory columns rather than the timing ones there.

In [ ]:
dp_seconds = run(dp_trainer, STEPS)
print(f"replicated (fsdp_size=1): {dp_seconds / STEPS * 1000:.0f} ms/step,"
      f" {device_memory() / 1e9:.2f} GB on device 0")

fsdp_seconds = run(fsdp_trainer, STEPS)
print(f"sharded (fsdp_size=8):    {fsdp_seconds / STEPS * 1000:.0f} ms/step,"
      f" {device_memory() / 1e9:.2f} GB on device 0")

## Checkpoints cross meshes

`dp_trainer` wrote its final state under `./checkpoints/scaling/lm-replicated`. The trainer below is built with `fsdp_size=8`, so its state lives sharded, and it restores that checkpoint into its own layout. The restore template comes from the freshly initialised state of this run, and the checkpoint's shards are reassembled onto whatever mesh asks for them. The five steps it then runs show that what arrived is a model that trains, not just a tree of the right shapes.

In [ ]:
restored = make_trainer(8, "lm-sharded-restore",
                        load_from="./checkpoints/scaling/lm-replicated")
run(restored, 5)
print("restored onto the fsdp mesh and trained 5 more steps")

## What changes on a multi-host pod

Everything above was one process. On a TPU pod slice every host runs the same script and sees the whole slice, and the only new obligations are joining the process pool before anything else touches JAX, and data and checkpoints that all hosts can reach:

```python
from dew.training import prepare_process

prepare_process("flip_only", multi_host=True)   # joins the pool from the cluster env
```

`prepare_process` calls `jax.distributed.initialize()`, which finds the coordinator from the environment a TPU pod provides. The data pipeline shards records by process, so each host reads its own part of the dataset with no coordination, and checkpoints go straight from every device to a bucket with `--trainer.checkpoint-fs gcs`.

The `dew-tpu` command drives all of it from your laptop:

```bash
dew-tpu create my-slice --type v5e-8     # create the slice, wait for READY
dew-tpu setup my-slice --from-source      # install dew and jax[tpu] on every worker
dew-tpu train my-slice --job lm-1 -- \
    python recipes/lm/train.py --data.dataset /mnt/gcs-data/shakespeare \
    --trainer.multi-host True --trainer.fsdp-size 8
```

`train` syncs the working tree to every worker, starts the recipe on all of them detached under one job name, and follows worker 0's log; `dew-tpu logs my-slice lm-1 --follow` comes back to it. Nothing in the recipe changes between one host and eight. The same `fsdp_size` knob picks the mesh, and the batch and the shards are per-process pieces of one global run.

## Where to go next

On a slice larger than one host, the `data` axis grows past 8 with `fsdp_size` held at 8, or both grow; `build_mesh` divides whatever device count the pool exposes. The threshold knob `fsdp_min_param_size` decides which parameters are worth splitting. On the model here it leaves only the norm scales replicated, and on a decoder with a large vocabulary the embedding table is the first thing worth sharding. Gradient accumulation (`grad_accum_steps`) trades step time for a bigger effective batch when the batch no longer fits in device memory.